# 🤖 ขั้นตอนที่ 4: Modeling
### (การสร้างและฝึกสอนโมเดล Machine Learning)
---
**วัตถุประสงค์ในขั้นตอนนี้:**
1. แบ่งชุดข้อมูล Train / Test ตามลำดับเวลา (Time-based Splitting 80/20)
2. สร้างโมเดล `MultiOutputClassifier(LogisticRegression)`
3. ฝึกสอนโมเดลเพื่อทำนาย 2 ตลาดพร้อมกัน (Asian Handicap & Over/Under)
4. บันทึกโมเดลไว้ในโฟลเดอร์ `models/`

## 1. นำเข้าไลบรารี Machine Learning

In [ ]:
import os, sys
# จัดการ Working Directory ให้ชี้ไปที่โฟลเดอร์หลักของโปรเจกต์เสมอ
if not os.path.exists('data') and os.path.exists('../data'):
    os.chdir('..')
if os.getcwd() not in sys.path:
    sys.path.insert(0, os.getcwd())
import joblib
import pandas as pd
import numpy as np
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.multioutput import MultiOutputClassifier

print("✅ นำเข้าไลบรารี Machine Learning สำเร็จ")

## 2. โหลดชุดข้อมูลที่ผ่านการ Preprocessing จากขั้นตอนที่ 3

In [ ]:
df = pd.read_csv('data/preprocessed_matches.csv')
print(f"โหลดข้อมูลสำเร็จ: {df.shape[0]} แถว, {df.shape[1]} คอลัมน์")

# โหลดรายชื่อ Feature Columns 47 ตัวของระบบ
model_meta = joblib.load('models/standalone_model.pkl')
feature_cols = model_meta['feature_columns']

X = df[feature_cols].copy()
y = df[['ah_label', 'ou_label']].astype(int)
print(f"Matrix ฟีเจอร์ X: {X.shape}, ป้ายกำกับเป้าหมาย y: {y.shape}")

## 3. การแบ่งชุดข้อมูลฝึกสอนและชุดทดสอบ (Train / Test Split)

In [ ]:
# แบ่งข้อมูล 80% เป็น Train และ 20% หลังสุดเป็น Test ตามลำดับเวลา
split_idx = int(len(X) * 0.8)
X_train, X_test = X.iloc[:split_idx], X.iloc[split_idx:]
y_train, y_test = y.iloc[:split_idx], y.iloc[split_idx:]

print(f"จำนวนแถวชุดฝึกสอน (Train): {len(X_train)} นัด")
print(f"จำนวนแถวชุดทดสอบ (Test):   {len(X_test)} นัด")

## 4. สร้างและฝึกสอนโมเดล (Model Training)

In [ ]:
# ใช้ Pipeline ที่มี Imputer + Scaler + LogisticRegression แบบเดียวกับระบบหลัก
base_estimator = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler()),
    ('clf', LogisticRegression(C=1.0, max_iter=4000, random_state=42))
])
model = MultiOutputClassifier(base_estimator)

# ฝึกสอนโมเดลบน X_train
model.fit(X_train, y_train)
print("✅ ฝึกสอนโมเดล (Model Training) เสร็จสมบูรณ์แล้ว")

## 5. บันทึกโมเดลเก็บไว้ใช้งาน (Save Trained Model)

In [ ]:
save_payload = {
    'model': model,
    'feature_columns': feature_cols
}
joblib.dump(save_payload, 'models/student_model.pkl')
print("✅ บันทึกโมเดลไว้ที่ models/student_model.pkl สำเร็จ")